In [1]:

import fastai
import torch

from pprint import pprint
from pathlib import Path

from omegaconf import OmegaConf

from fastai.text.all import *

from Configs import MAX_SEQUENCE_LENGTH
from GrooveModel import Datasets, CollateFunctions
from GrooveModel.Tokenizer import Tokenizer
from GrooveModel.Embedding.Embedding import MultiTaskDNAEmbedding
from GrooveModel.Models import MultiTaskDNAxLSTM, MultiTaskDNAModelConfig


In [2]:
# Paths
BASE_PATH = Path.cwd().parent
DATA_PATH = BASE_PATH / 'Data'
DNA_PATH = DATA_PATH / 'dnas.json'
MODEL_PATH = BASE_PATH / 'GrooveModel'
XLSTM_PATH = MODEL_PATH / 'xlstm'

In [3]:
ds_config_string = f"""
dna_path: {DNA_PATH}
# ADD more params like subset, training/validation/testing,...
convert_to_tensor: True
"""

# Final OmegaConf config
ds_conf = OmegaConf.create(ds_config_string)


In [4]:
# Dataset
dna_ds = Datasets.DNANextTokenDataset(ds_conf, 'train', tokenizer=Tokenizers.MultiTaskDnaTokenizer)
len(dna_ds)

622

In [5]:
dna_ds[0]

(tensor([[  3, 128,   1,  61,   3, 161,   1],
         [  3, 128,   2,  61,   3, 161,   1],
         [  2, 128,   3,  61,   3, 161,   1],
         [  2, 128,   4,  81,   3, 161,   1],
         [  2, 128,   5,  63,   3, 161,   1],
         [  2, 128,   6,  40,   3, 161,   1],
         [  4, 128,   6,  84,   3, 161,   1],
         [  3, 128,   7,  61,   3, 161,   1],
         [  3, 125,   8,  61,   3, 161,   1],
         [  2, 128,   9,  60,   3, 161,   1],
         [  2, 128,  10,  80,   3, 161,   1],
         [  2, 128,  11,  62,   3, 161,   1],
         [  2, 128,  12,  39,   3, 161,   1],
         [  4, 128,  12,  83,   3, 161,   1],
         [  3, 128,  13,  60,   3, 161,   1],
         [  1,   1,  14,  61,   3, 161,   1],
         [  1,   1,  15,  61,   3, 161,   1],
         [  1,   1,  16,  61,   3, 161,   1],
         [  2, 128,   1,  61,   3, 161,   1],
         [  7, 128,   1,  61,   3, 161,   1],
         [  1,   1,   2,  61,   3, 161,   1],
         [  1,   1,   3,  61,   3,

In [6]:
Tokenizers.MultiDnaToken.from_tensor(dna_ds[0][0][0])

DNAToken(Instrument=3, Velocity=128, BeatUnit=1, BeatUnitOffset=61, GridFactor=3, Bpm=161, TimeSignature=1)

In [7]:
# DATALOADER

dataloader = torch.utils.data.DataLoader(dna_ds, batch_size=2, shuffle=True, collate_fn=CollateFunctions.pad_batch)

# Pad zeroes at end:
# The most powerful attribute of LSTMs and RNNs in general is that their parameters are shared along the time frames(Parameters recur over time frames) but the parameter sharing relies upon the assumption that the same parameters can be used for different time steps i.e. the relationship between the previous time step and the next time step does not depend on t as explained here in page 388, 2nd paragraph.
#
# In short, padding zeros at the end, theoretically should not change the accuracy of the model. I used the adverb theoretically because at each time step LSTM's decision depends on its cell state among other factors and this cell state is kind of a short summary of the past frames. As far as I understood, that past frames may be missing in your case. I think what you have here is a little trade-off.
#
# I would rather pad zeros at the end because it doesn't completely conflict with the underlying assumption of RNNs and it's more convenient to implement and keep track of.
#
# On the implementation side, I know tensorflow calculates the loss function once you give it the sequences and the actual sequence size of each sample(e.g. for 4 5 6 7 0 0 0 0 0 0 you also need to give it the actual size which is 4 here) assuming you're implementing the option 2. I don't know whether there is an implementation for option 1, though.

padded_inputs, padded_targets, lengths = next(iter(dataloader))
pprint('padded input shape:')
pprint(padded_inputs.shape)
pprint('padded target shape:')
pprint(padded_targets.shape)
pprint('lengths:')
pprint(lengths)
pprint('padded inputs:')
pprint(padded_inputs)
pprint('padded targets:')
pprint(padded_targets)

'padded input shape:'
torch.Size([2, 347, 7])
'padded target shape:'
torch.Size([2, 347, 7])
'lengths:'
tensor([171, 347])
'padded inputs:'
tensor([[[  2, 128,   1,  ...,   3, 141,   1],
         [  7, 128,   1,  ...,   3, 141,   1],
         [  2, 128,   2,  ...,   3, 141,   1],
         ...,
         [  2, 128,  11,  ...,   3, 141,   1],
         [  2, 128,  12,  ...,   3, 141,   1],
         [  3, 128,  13,  ...,   3, 141,   1]],

        [[  2, 128,   1,  ...,   3, 151,   1],
         [  7, 128,   1,  ...,   3, 151,   1],
         [  2, 128,   2,  ...,   3, 151,   1],
         ...,
         [  0,   0,   0,  ...,   0,   0,   0],
         [  0,   0,   0,  ...,   0,   0,   0],
         [  0,   0,   0,  ...,   0,   0,   0]]], dtype=torch.int32)
'padded targets:'
tensor([[[  7, 128,   1,  ...,   3, 141,   1],
         [  2, 128,   2,  ...,   3, 141,   1],
         [  2, 128,   3,  ...,   3, 141,   1],
         ...,
         [  2, 128,  12,  ...,   3, 141,   1],
         [  3, 128,  13, 

In [8]:
from GrooveModel.Embedding.Embedding import MultiTaskDNAEmbeddingConfig

# embedding dimensions
instrument_embedding_dim = 4 # 7 values
velocity_embedding_dim = 22 # 128 values
offset_embedding_dim = 16 # 120 values
time_signature_embedding_dim = 8 # 40 values
grid_embedding_dim = 4 # 5 values
bpm_embedding_dim = 16 # 300 values
beat_unit_embedding_dim = 26 # 72 - 1024 values


embedding_config_string = f"""
instruments:
    embedding_dim: {instrument_embedding_dim}
velocities:
    embedding_dim: {velocity_embedding_dim}
offsets:
    embedding_dim: {offset_embedding_dim}
time_signature:
    embedding_dim: {time_signature_embedding_dim}
grid_factor:
    embedding_dim: {grid_embedding_dim}
bpm:
    embedding_dim: {bpm_embedding_dim}
beat_units:
    embedding_dim: {beat_unit_embedding_dim}
    absolute_beat_units: False
"""
# xLSTM already does layer norm

# 1. Define the schema
schema = OmegaConf.structured(MultiTaskDNAEmbeddingConfig())

# 2. Parse the string into a config
parsed_config = OmegaConf.create(embedding_config_string)

# 3. Merge schema with actual config
embedding_config = OmegaConf.merge(schema, parsed_config)

In [9]:
embedder = MultiTaskDNAEmbedding(embedding_config)
embedder

MultiTaskDNAEmbedding(
  (sub_embeddings): ModuleDict(
    (instrument): Embedding(8, 4, padding_idx=0)
    (velocity): Embedding(129, 22, padding_idx=0)
    (beat_unit): Embedding(73, 26, padding_idx=0)
    (offset): Embedding(121, 16, padding_idx=0)
    (grid): Embedding(6, 4, padding_idx=0)
    (bpm): Embedding(301, 16, padding_idx=0)
    (time_signature): Embedding(13, 8, padding_idx=0)
  )
)

In [18]:
test_embedding = embedder(padded_inputs)
test_embedding

tensor([[[ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         [-0.0687, -0.6566,  0.1844,  ...,  0.6700, -0.1581,  1.3466],
         [ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         ...,
         [ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         [ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         [-0.2465, -0.6864, -1.6051,  ...,  0.6700, -0.1581,  1.3466]],

        [[ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         [-0.0687, -0.6566,  0.1844,  ...,  0.6700, -0.1581,  1.3466],
         [ 1.0934, -0.0979,  0.9624,  ...,  0.6700, -0.1581,  1.3466],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<CatBackward0>)

In [11]:
embedder.sub_embeddings['instrument'].weight

Parameter containing:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.4731, -0.5316,  0.4786,  0.9572],
        [ 1.0934, -0.0979,  0.9624,  1.7639],
        [-0.2465, -0.6864, -1.6051,  0.0971],
        [ 0.5760,  0.8543, -0.7619,  1.7237],
        [-0.3526,  0.3269, -1.7985, -0.3048],
        [ 0.5178,  0.0566,  1.3138,  1.7181],
        [-0.0687, -0.6566,  0.1844,  0.6113]], requires_grad=True)

In [12]:
context_length = MAX_SEQUENCE_LENGTH
num_heads = embedder.embedding_dim // 16
num_heads

6

In [28]:
from omegaconf import OmegaConf
from dacite import from_dict, Config as DaciteConfig

# Define raw YAML
xlstm_cfg = f"""
mlstm_block:
  mlstm:
    conv1d_kernel_size: 4
    qkv_proj_blocksize: 4
    num_heads: 4
slstm_block:
  slstm:
    backend: {'cuda' if torch.cuda.is_available() else 'vanilla'}
    num_heads: 4
    conv1d_kernel_size: 4
    bias_init: powerlaw_blockdependent
  feedforward:
    proj_factor: 1.3
    act_fn: gelu
slstm_at: [1]
context_length: {context_length}
num_blocks: 7
embedding_dim: {embedder.embedding_dim}
add_embedding_dropout: True
dropout: 0.1
"""

# Load into OmegaConf
cfg_omega = OmegaConf.create(xlstm_cfg)

# Convert to dataclass
model_config = from_dict(
    data_class=MultiTaskDNAModelConfig,
    data=OmegaConf.to_container(cfg_omega, resolve=True),
    config=DaciteConfig(strict=True)
)

assert model_config.embedding_dim % model_config.slstm_block.slstm.num_heads == 0, \
    f"embedding_dim ({model_config.embedding_dim}) must be divisible by num_heads ({model_config.slstm_block.slstm.num_heads})"
pprint(model_config)

MultiTaskDNAModelConfig(mlstm_block=mLSTMBlockConfig(mlstm=mLSTMLayerConfig(proj_factor=2.0,
                                                                            round_proj_up_dim_up=True,
                                                                            round_proj_up_to_multiple_of=64,
                                                                            _proj_up_dim=192,
                                                                            conv1d_kernel_size=4,
                                                                            qkv_proj_blocksize=4,
                                                                            num_heads=4,
                                                                            embedding_dim=96,
                                                                            bias=False,
                                                                            dropout=0.1,
                                             

In [29]:
model = MultiTaskDNAxLSTM(model_config, embedding_config)

In [30]:
pprint(model.forward(padded_inputs))

{'beat_unit': tensor([[[-0.3454, -0.9394, -0.4439,  ..., -0.8337, -0.3555, -0.2030],
         [-0.3517, -0.9860, -0.1923,  ..., -0.5971, -0.5001, -0.2918],
         [-0.7132, -1.0763, -0.2982,  ..., -0.6476,  0.8463,  0.1967],
         ...,
         [-0.6216,  0.4374,  0.1543,  ..., -0.7304,  0.1001,  0.0922],
         [-0.3374, -0.6301, -0.3741,  ..., -0.9093,  0.4222, -0.2462],
         [-1.2406, -1.1076, -0.1115,  ..., -0.0401,  0.1374, -0.2248]],

        [[-0.3726, -0.5209,  0.2000,  ..., -0.8260,  0.1506, -0.1249],
         [-0.0440, -0.7257,  0.5202,  ..., -0.5173,  0.3711, -0.6215],
         [-0.1485, -0.5541, -0.2519,  ..., -0.3448,  0.2064,  0.4626],
         ...,
         [ 0.0193, -0.3798,  0.2119,  ...,  1.7492,  0.3673, -1.1841],
         [ 0.1692, -0.3957,  0.1797,  ...,  1.7414,  0.4398, -1.2218],
         [ 0.0307, -0.4874,  0.1225,  ...,  1.7983,  0.3075, -1.2213]]],
       grad_fn=<UnsafeViewBackward0>),
 'bpm': tensor([[[-0.4534, -0.5393, -1.4012,  ...,  0.2246, -0.

In [11]:
# TODO:
# 1. BeatUnit.py DONE
# 2. Tokenizer still needs to utilise all the encode functions DONE
# 3. absolute/relative/playing_together beat_units DONE
# 4. Add handling to embedding DONE
# 5. Rework xLSTM DONE
# 6. Create Learner environment, metrics, callbacks

In [21]:
# split beat/fill ? // OPEN
# create embeddings for all required dna values // DONE
# conat them // DONE
# positional embedding according to gridunit (beat position) // DONE
# feed to xlstm
# split output tensor
# reconstruct dna unit

# Rewrite all into python files




In [ ]:
# DONT FORGET EMBEDDING NORMALIZATION
# MAYBE POOLING OF EMBEDDINGS?

1. Data Preprocessing will require a step of preforming the file read layout (mid + text)
2. Make a class for preparing midi files and their meta info to dna-layout
3. Make class abstract and define specific readers for each source (foundational vs. lmd vs. basti ,...)
4. Run json extractor
5. Create dataset with json
6. create tokens
7. embed tokens
8. ...


# DOCUMENT AFTER TEST!!!!!!!!!!!